In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/harsh/Data-engineering-zoomcamp/batch_processing_spark/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/18 12:53:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.version

'4.2.0'

In [8]:
df_hw = spark.read.parquet('data/HW/yellow_tripdata_2025-11.parquet')

In [9]:
df_hw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df_hw = df_hw.repartition(4)

In [12]:
df_hw.write.parquet('data/HW/pq/')

In [15]:
df_hw.createOrReplaceTempView('hw_data')

In [16]:
df_hw.show(10)

[Stage 9:============================================>              (3 + 1) / 4]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [20]:
spark.sql("""
SELECT
    DATE(tpep_pickup_datetime) AS taxi_trips,
    COUNT(1) AS trip_count
FROM
    hw_data
WHERE 
    DATE(tpep_pickup_datetime) >= '2025-11-15' 
    AND DATE(tpep_pickup_datetime) < '2025-11-16'
GROUP BY 
    DATE(tpep_pickup_datetime)
""").show()

+----------+----------+
|taxi_trips|trip_count|
+----------+----------+
|2025-11-15|    162604|
+----------+----------+



In [21]:
spark.sql("""
SELECT
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600 AS trip_duration_hours
FROM
    hw_data
ORDER BY 
    trip_duration_hours DESC
LIMIT 1
""").show()

[Stage 32:>                                                         (0 + 4) / 4]

+--------+--------------------+---------------------+-------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_hours|
+--------+--------------------+---------------------+-------------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|  90.64666666666666|
+--------+--------------------+---------------------+-------------------+



In [28]:
df_zones = spark.read \
    .option("header", "true") \
      .option("inferSchema", "true") \
    .csv('data/HW/taxi_zone_lookup.csv')

In [29]:
df_zones.head(5)

[Row(LocationID=1, Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID=2, Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID=3, Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(LocationID=4, Borough='Manhattan', Zone='Alphabet City', service_zone='Yellow Zone'),
 Row(LocationID=5, Borough='Staten Island', Zone='Arden Heights', service_zone='Boro Zone')]

In [30]:
df_zones.createOrReplaceTempView("zone_lookup")

In [32]:


spark.sql("""
SELECT
    z.Zone,
    COUNT(1) AS pickup_count
FROM
    hw_data h
JOIN
    zone_lookup z
ON
    h.PULocationID = z.LocationID
GROUP BY
    z.Zone
ORDER BY
    pickup_count ASC
""").show()

[Stage 55:>                                                         (0 + 4) / 4]

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|Eltingville/Annad...|           1|
|Governor's Island...|           1|
|       Arden Heights|           1|
|       Port Richmond|           3|
|   Rossville/Woodrow|           4|
|         Great Kills|           4|
| Green-Wood Cemetery|           4|
|       Rikers Island|           4|
|         Jamaica Bay|           5|
|         Westerleigh|          12|
|New Dorp/Midland ...|          14|
|       West Brighton|          14|
|             Oakwood|          14|
|        Crotona Park|          14|
|       Willets Point|          15|
|Breezy Point/Fort...|          16|
|Saint George/New ...|          17|
|       Broad Channel|          18|
|     Mariners Harbor|          21|
|Heartland Village...|          22|
+--------------------+------------+
only showing top 20 rows
